# MediKiosk - Fine-Tuning Llama 3.1 8B with Unsloth / QLoRA
This notebook fine-tunes **Meta-Llama-3.1-8B-Instruct** using Unsloth (2x faster, 70% less VRAM) on the curated **AyurGenixAI** clinical dataset for structured Ayurvedic clinical case extraction in MediKiosk.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets triton

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection (Float16 for Tesla T4/V100, Bfloat16 for Ampere+)
load_in_4bit = True # 4bit quantization to fit in 16GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Add LoRA adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# Load formatted dataset (train.jsonl generated via dataset_formatter.py)
dataset = load_dataset("json", data_files={"train": "train.jsonl"}, split="train")

# Alpaca formatting function with Llama 3.1 prompt template
llama3_prompt = """<|start_header_id|>system<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>

{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = llama3_prompt.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample formatted record:")
print(dataset[0]["text"][:1000])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Test Model Inference on a fresh patient narrative
FastLanguageModel.for_inference(model)

test_prompt = llama3_prompt.format(
    instruction = dataset[0]["instruction"],
    input = "Patient reports: severe throbbing headache on right side, nausea, sensitivity to bright lights.\nAdditional context: severity appears severe; expected duration: 3 days.\nPatient profile: 32 years, Female.\nLifestyle factors: sleep: poor sleep, stress: high stress.",
    output = ""
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Save LoRA adapter locally & export to GGUF / Hugging Face
model.save_pretrained("medikiosk_llama3_lora")
tokenizer.save_pretrained("medikiosk_llama3_lora")
print("Model saved to medikiosk_llama3_lora/")